# Preprocessing — Lab 3 complexity audit

This notebook is the complexity analysis for the **preprocessing** module. Each nonempty source line appears below with an upper-bound or a note about its enclosing operation. The costs refer to the entire execution of that statement over enclosing loops; lines that form a multiline expression must be interpreted together. Detailed globally combined bounds are in `../../docs/complexity_summary.md`.

**Symbols.** `A` actors; `C=2` conditions; `R=C·A` recordings; `Z` ZIP entries; `N` samples; `K` 25 ms frame samples; `H` 10 ms hop samples; `W≈N/H` frames per recording; `W0` longer pYIN frames; `F=5` characteristics; `S=6` statistics; `d=4` classifier inputs; `M=2` models; `Pixels` number of saved image pixels; `T_pYIN` and `T_fit` are deliberately opaque library-level costs.

## Line-by-line analysis: `audio.py`

| Line | Source statement | Estimated cost / reason |
|---:|---|---|
| 1 | `"""Data ingestion and preprocessing for paired RAVDESS emotional speech."""` | Comment/docstring — no runtime work |
| 3 | `from __future__ import annotations` | O(1) declaration/binding; function call cost discussed below |
| 5 | `from io import BytesIO` | O(1) declaration/binding; function call cost discussed below |
| 6 | `from pathlib import Path` | O(1) declaration/binding; function call cost discussed below |
| 7 | `import re` | O(1) declaration/binding; function call cost discussed below |
| 8 | `import zipfile` | O(1) declaration/binding; function call cost discussed below |
| 10 | `import numpy as np` | O(1) declaration/binding; function call cost discussed below |
| 11 | `import soundfile as sf` | O(1) declaration/binding; function call cost discussed below |
| 13 | `EMOTION_CODES = {"Neutral": "01", "Sad": "04"}` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 14 | `_RAVDESS_RE = re.compile(r"^(\d\d)-(\d\d)-(\d\d)-(\d\d)-(\d\d)-(\d\d)-(\d\d)\.wav$", re.I)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 17 | `def parse_filename(name: str) -> dict[str, str] \| None:` | O(1) declaration/binding; function call cost discussed below |
| 18 | `"""Parse RAVDESS identifiers or return None for nonmatching filenames."""` | Comment/docstring — no runtime work |
| 19 | `match = _RAVDESS_RE.fullmatch(Path(name).name)` | O(m) for a filename of length m |
| 20 | `if match is None:` | O(1) condition/control unless evaluating an included collection |
| 21 | `return None` | O(1) to return references (ignoring any nested expression) |
| 22 | `return dict(zip(("modality", "channel", "emotion", "intensity", "statement", "repetition", "actor"), match.groups...` | O(1) to return references (ignoring any nested expression) |
| 25 | `def choose_matched_members(` | O(1) declaration/binding; function call cost discussed below |
| 26 | `names: list[str],` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 27 | `n_actors: int \| None = 24,` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 28 | `statement: str = "01",` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 29 | `repetition: str = "01",` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 30 | `intensity: str = "01",` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 31 | `) -> tuple[list[str], dict[str, list[str]]]:` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 32 | `"""Select one neutral and one sad speech WAV for each matched actor.` | Comment/docstring — no runtime work |
| 34 | `Uses filename metadata, not the 60-files-per-actor archive-order assumption.` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 35 | `The same statement, repetition, intensity, and actor are used in both groups.` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 36 | `"""` | Comment/docstring — no runtime work |
| 37 | `found: dict[str, dict[str, str]] = {}` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 38 | `for name in names:` | O(Z) archive entries scanned |
| 39 | `meta = parse_filename(name)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 40 | `if meta is None or meta["modality"] != "03" or meta["channel"] != "01":` | O(1) condition/control unless evaluating an included collection |
| 41 | `continue` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 42 | `if (meta["statement"], meta["repetition"], meta["intensity"]) != (statement, repetition, intensity):` | O(1) condition/control unless evaluating an included collection |
| 43 | `continue` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 44 | `for label, emotion_code in EMOTION_CODES.items():` | O(C) emotion-code matches |
| 45 | `if meta["emotion"] == emotion_code:` | O(1) condition/control unless evaluating an included collection |
| 46 | `found.setdefault(meta["actor"], {})[label] = name` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 48 | `actors = sorted(a for a, pairs in found.items() if all(k in pairs for k in EMOTION_CODES))` | O(A log A) actor-key sort |
| 49 | `if n_actors is not None:` | O(1) condition/control unless evaluating an included collection |
| 50 | `if n_actors < 1:` | O(1) condition/control unless evaluating an included collection |
| 51 | `raise ValueError("n_actors must be positive, or None for all matching actors")` | O(1) condition/control unless evaluating an included collection |
| 52 | `actors = actors[:n_actors]` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 53 | `if not actors:` | O(1) condition/control unless evaluating an included collection |
| 54 | `raise ValueError("No paired neutral/sad speech records found; verify archive and RAVDESS filename codes")` | O(1) condition/control unless evaluating an included collection |
| 55 | `return actors, {condition: [found[actor][condition] for actor in actors] for condition in EMOTION_CODES}` | O(1) to return references (ignoring any nested expression) |
| 58 | `def read_wav_in_memory(archive: zipfile.ZipFile, member: str) -> tuple[np.ndarray, int]:` | O(1) declaration/binding; function call cost discussed below |
| 59 | `"""Decode one ZIP-contained WAV without writing a temporary WAV to disk."""` | Comment/docstring — no runtime work |
| 60 | `audio_bytes = archive.read(member)` | O(L_file) decompression of this file |
| 61 | `signal, fs = sf.read(BytesIO(audio_bytes), dtype="float32", always_2d=True)` | O(N) decoding per file; O(R·N) for all files |
| 62 | `mono = signal.mean(axis=1)  # Voice is analysed in mono.` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 63 | `return mono, int(fs)` | O(1) to return references (ignoring any nested expression) |
| 66 | `def load_paired_audios(zip_path: str \| Path, n_actors: int \| None = 24) -> tuple[list[str], dict[str, list[np.n...` | O(1) declaration/binding; function call cost discussed below |
| 67 | `"""Load matched neutral/sad recordings from a ZIP directly into memory.` | Comment/docstring — no runtime work |
| 69 | `Returns actor ids, two lists of 1-D arrays, and a shared sample rate.` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 70 | `"""` | Comment/docstring — no runtime work |
| 71 | `with zipfile.ZipFile(zip_path) as archive:` | O(1) condition/control unless evaluating an included collection |
| 72 | `actors, members = choose_matched_members(archive.namelist(), n_actors)` | O(Z) enumeration |
| 73 | `groups: dict[str, list[np.ndarray]] = {}` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 74 | `rates: set[int] = set()` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 75 | `for condition, filenames in members.items():` | O(C) emotional conditions |
| 76 | `groups[condition] = []` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 77 | `for filename in filenames:` | O(R) filenames across groups |
| 78 | `samples, fs = read_wav_in_memory(archive, filename)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 79 | `groups[condition].append(samples)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 80 | `rates.add(fs)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 81 | `if len(rates) != 1:` | O(1) condition/control unless evaluating an included collection |
| 82 | `raise ValueError("Audio sample rates differ; explicitly resample before comparison")` | O(1) condition/control unless evaluating an included collection |
| 83 | `return actors, groups, rates.pop()` | O(1) to return references (ignoring any nested expression) |
| 86 | `def truncate_matched_groups(groups: dict[str, list[np.ndarray]]) -> dict[str, np.ndarray]:` | O(1) declaration/binding; function call cost discussed below |
| 87 | `"""Truncate all groups to one shared sample count, returning 2D matrices."""` | Comment/docstring — no runtime work |
| 88 | `records = [audio for audios in groups.values() for audio in audios]` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 89 | `if not records:` | O(1) condition/control unless evaluating an included collection |
| 90 | `raise ValueError("No recordings available")` | O(1) condition/control unless evaluating an included collection |
| 91 | `target_length = min(map(len, records))` | O(R) lengths scanned |
| 92 | `if target_length == 0:` | O(1) condition/control unless evaluating an included collection |
| 93 | `raise ValueError("Empty recording found")` | O(1) condition/control unless evaluating an included collection |
| 94 | `return {condition: np.stack([audio[:target_length] for audio in audios]) for condition, audios in groups.items()}` | O(R·N) copying truncated samples |
| 97 | `def load_members_via_disk(zip_path: str \| Path, members: list[str], temp_dir: str \| Path) -> list[np.ndarray]:` | O(1) declaration/binding; function call cost discussed below |
| 98 | `"""Reference ingestion implementation: extract each WAV before reading it."""` | Comment/docstring — no runtime work |
| 99 | `with zipfile.ZipFile(zip_path) as archive:` | O(1) condition/control unless evaluating an included collection |
| 100 | `paths = [archive.extract(member, path=temp_dir) for member in members]` | O(L_file) decompression plus disk write |
| 101 | `return [sf.read(path, dtype="float32", always_2d=True)[0].mean(axis=1) for path in paths]` | O(N) decoding per file; O(R·N) for all files |

## Global algorithm bounds

| Algorithm | End-to-end bound |
|---|---|
| `parse_filename` | O(m) time and O(1) output for fixed RAVDESS fields; m is filename length. |
| `choose_matched_members` | O(Z·m + A log A) time; O(Z + A) transient storage; Z archive members. |
| `read_wav_in_memory` | O(L_file + N) time and memory due to decompression and mono conversion. |
| `load_paired_audios` | O(Z + A log A + L + R·N) time; O(R·N) memory for decoded samples. |
| `truncate_matched_groups` | O(R·N) time and output memory; O(R) to find common length. |
| `load_members_via_disk (reference)` | O(L + R·N) time, plus O(L) disk writes; both methods have the same big-O. |

**Optimization.** Unlike the original disk-extraction approach and fixed 60-recording offsets, in-memory ZIP decoding avoids temporary file writes and metadata parsing correctly pairs actor/condition. The **asymptotic I/O work remains linear**, while measured constant factors may improve.

In [ ]:
# Summary of the module: view the registered algorithm bounds without loading data.
complexities = [('parse_filename', 'O(m) time and O(1) output for fixed RAVDESS fields; m is filename length.'), ('choose_matched_members', 'O(Z·m + A log A) time; O(Z + A) transient storage; Z archive members.'), ('read_wav_in_memory', 'O(L_file + N) time and memory due to decompression and mono conversion.'), ('load_paired_audios', 'O(Z + A log A + L + R·N) time; O(R·N) memory for decoded samples.'), ('truncate_matched_groups', 'O(R·N) time and output memory; O(R) to find common length.'), ('load_members_via_disk (reference)', 'O(L + R·N) time, plus O(L) disk writes; both methods have the same big-O.')]
for name, complexity in complexities:
    print(f"{name}: {complexity}")